# Figure 2 — perturbation programs across sequencing- and imaging-based screens

Every code cell below is copied verbatim from an existing analysis notebook; nothing
here is new analysis. Each source section starts with a small `[reproducibility glue]`
cell that changes into the source notebook's folder (so its relative data and results
paths still work) and resets the sherlock config. Cells that are not labelled with a
panel are the prerequisites that panel needed in the source notebook (data loading,
model loading, helper definitions). Panels are produced separately rather than laid out
as in the manuscript. Anything that could not be found in the current notebooks is
marked **FLAG**.

In [ ]:
import os
REPRO_DIR = os.path.abspath('')

## Source: `notebooks/raefish/raefish_run.ipynb`

Cells copied verbatim (source cell indices): 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29. Training (14) and saving (15) are not copied; cell 16 loads `results/raefish.pth`.

In [ ]:
# [reproducibility glue] run from the source notebook's directory so its relative paths resolve
os.chdir(os.path.join(REPRO_DIR, '../notebooks/raefish'))
if 'slk' in globals():
    slk.configs.reset_config()

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import sys

import seaborn as sns
import matplotlib.pyplot as plt

import torch

#Make sure to path to wherever sherlock is located
sys.path.append('../../..')
import sherlock as slk

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
slk.configs.show()
slk.configs.set_config('ntc_label', 'non-targeting')
print('\n')
slk.configs.show()
slk.configs.reset_config()
slk.configs.get_config('ntc_label')

In [ ]:
data = sc.read_h5ad("../../datasets/raefish/raefish.h5ad")
slk.pp.slk_prepare_data(data, pert_key="RaeFishGene", ntc_label="NonTargeting", treatment_key=None, inplace=True)

In [ ]:
# plot distribution of perturbations across datasets to check dataset-uniqueness
ct = pd.crosstab(data.obs['pert'], data.obs['dataset'])  # perturbation x dataset counts
datasets_per_pert = (ct > 0).sum(axis=1)

# summary
n_total = len(datasets_per_pert)
n_unique = (datasets_per_pert == 1).sum()
print(f"{n_unique} / {n_total} perturbations ({n_unique/n_total:.1%}) occur in exactly one dataset")

# figure: histogram of how many datasets each perturbation appears in + heatmap of top perturbations
topN = 50
top_pert = ct.sum(axis=1).sort_values(ascending=False).head(topN).index
ct_top = ct.loc[top_pert]
ct_top_frac = ct_top.divide(ct_top.sum(axis=1), axis=0).fillna(0)  # row-normalized fractions

fig, axes = plt.subplots(1, 2, figsize=(18, 10), gridspec_kw={'width_ratios': [1, 1.8]})

# histogram
axes[0].hist(datasets_per_pert, bins=range(1, datasets_per_pert.max() + 2), color='C0', edgecolor='k', align='left')
axes[0].set_xlabel('Number of datasets per perturbation')
axes[0].set_ylabel('Number of perturbations')
axes[0].axvline(1, color='red', linestyle='--', label='unique (1 dataset)')
axes[0].legend()
axes[0].set_title('How many datasets contain each perturbation')

# heatmap for top perturbations (row-normalized)
sns.heatmap(ct_top_frac, ax=axes[1], cmap='viridis', cbar_kws={'label': 'Fraction of cells in dataset'},
            yticklabels=True)
axes[1].set_ylabel('Perturbation (top by count)')
axes[1].set_xlabel('Dataset')
axes[1].set_title(f'Top {topN} perturbations: dataset composition (row-normalized)')

plt.tight_layout()
plt.show()

In [ ]:
# plot number of cells per perturbation: histogram + top N barplot
pert_counts = ct.sum(axis=1)  # ct is perturbation x dataset counts

fig, axes = plt.subplots(1, 2, figsize=(18, 8), gridspec_kw={'width_ratios': [1, 2]})

# histogram of counts
axes[0].hist(pert_counts.values, bins=50, color='C0', edgecolor='k', alpha=0.8)
axes[0].set_xlabel('Cells per perturbation')
axes[0].set_ylabel('Number of perturbations')
axes[0].set_title('Distribution of cells per perturbation')
axes[0].axvline(pert_counts.median(), color='red', linestyle='--', label=f'median={int(pert_counts.median())}')
axes[0].legend()

# horizontal barplot of top N perturbations
topN = min(50, len(pert_counts))
top = pert_counts.sort_values(ascending=False).head(topN)
sns.barplot(x=top.values, y=top.index, palette='viridis', ax=axes[1])
axes[1].set_xlabel('Number of cells')
axes[1].set_ylabel('Perturbation')
axes[1].set_title(f'Top {topN} perturbations by cell count')

plt.tight_layout()
plt.show()

In [ ]:
# Cross-tab of counts: rows=perturbations, cols=datasets
ct = pd.crosstab(data.obs['pert'], data.obs['dataset'])

# Number of perturbations unique to the dataset
datasets_per_pert = (ct > 0).sum(axis=1)              # in how many datasets each pert appears
unique_perts = datasets_per_pert[datasets_per_pert == 1].index
unique_perts_per_dataset = (ct.loc[unique_perts] > 0).sum(axis=0).reindex(ct.columns, fill_value=0)

# Number of total perturbations in the dataset
total_perts_per_dataset = (ct > 0).sum(axis=0).astype(int)

# Number of cells in the dataset
cells_per_dataset = ct.sum(axis=0).astype(int)

# Average number of cells per perturbation in the dataset
avg_cells_per_pert = (cells_per_dataset / total_perts_per_dataset.replace(0, np.nan)).fillna(0).round(2)

# Assemble a single summary table
summary = pd.DataFrame({
    "unique_perturbations": unique_perts_per_dataset.astype(int),
    "total_perturbations": total_perts_per_dataset.astype(int),
    "cells": cells_per_dataset.astype(int),
    "avg_cells_per_perturbation": avg_cells_per_pert
}).reindex(ct.columns)

print("Per-dataset summary:")
print(summary)


In [ ]:
data = data[data.obs.dataset == '241203_kraken']

In [ ]:
# plot number of cells per perturbation: histogram + top N barplot
ct = pd.crosstab(data.obs['pert'], data.obs['dataset'])
pert_counts = ct.sum(axis=1)  # ct is perturbation x dataset counts

fig, axes = plt.subplots(1, 2, figsize=(18, 8), gridspec_kw={'width_ratios': [1, 2]})

# histogram of counts
axes[0].hist(pert_counts.values, bins=50, color='C0', edgecolor='k', alpha=0.8)
axes[0].set_xlabel('Cells per perturbation')
axes[0].set_ylabel('Number of perturbations')
axes[0].set_title('Distribution of cells per perturbation')
axes[0].axvline(pert_counts.median(), color='red', linestyle='--', label=f'median={int(pert_counts.median())}')
axes[0].legend()

# horizontal barplot of top N perturbations
topN = min(50, len(pert_counts))
top = pert_counts.sort_values(ascending=False).head(topN)
sns.barplot(x=top.values, y=top.index, palette='viridis', ax=axes[1])
axes[1].set_xlabel('Number of cells')
axes[1].set_ylabel('Perturbation')
axes[1].set_title(f'Top {topN} perturbations by cell count')

plt.tight_layout()
plt.show()

In [ ]:
min_cells = 20

# Global counts per perturbation (across all datasets)
pert_counts_full = data.obs['pert'].value_counts()

# Keep perturbations with >= min_cells total cells (ignore per-dataset splits)
valid_perts = pert_counts_full[pert_counts_full >= min_cells].index

mask = data.obs['pert'].isin(valid_perts)
print(f"Keeping {mask.sum()} / {data.n_obs} cells ({mask.sum()/data.n_obs:.1%}) "
      f"from {len(valid_perts)} perturbations (>= {min_cells} cells total)")

# Subset
data = data[mask].copy()

# Recompute summaries
ct = pd.crosstab(data.obs['pert'], data.obs['dataset'])
pert_counts = ct.sum(axis=1).astype(int)
datasets_per_pert = (ct > 0).sum(axis=1).astype(int)


In [ ]:
# plot number of cells per perturbation: histogram + top N barplot
ct = pd.crosstab(data.obs['pert'], data.obs['dataset'])
pert_counts = ct.sum(axis=1)  # ct is perturbation x dataset counts

fig, axes = plt.subplots(1, 2, figsize=(18, 8), gridspec_kw={'width_ratios': [1, 2]})

# histogram of counts
axes[0].hist(pert_counts.values, bins=50, color='C0', edgecolor='k', alpha=0.8)
axes[0].set_xlabel('Cells per perturbation')
axes[0].set_ylabel('Number of perturbations')
axes[0].set_title('Distribution of cells per perturbation')
axes[0].axvline(pert_counts.median(), color='red', linestyle='--', label=f'median={int(pert_counts.median())}')
axes[0].legend()

# horizontal barplot of top N perturbations
topN = min(50, len(pert_counts))
top = pert_counts.sort_values(ascending=False).head(topN)
sns.barplot(x=top.values, y=top.index, palette='viridis', ax=axes[1])
axes[1].set_xlabel('Number of cells')
axes[1].set_ylabel('Perturbation')
axes[1].set_title(f'Top {topN} perturbations by cell count')

plt.tight_layout()
plt.show()

In [ ]:
import scipy.sparse as sps

# fallback: sum across expression matrix
lib_size = np.asarray(data.X.sum(axis=1)).ravel()

# 2) number of cells per gene (non-zero counts)
X = data.X
if sps.issparse(X):
    cells_per_gene = np.asarray((X > 0).sum(axis=0)).ravel()
else:
    cells_per_gene = (X > 0).sum(axis=0).astype(int)

# 3) Plotting
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Library size histogram
axes[0].hist(np.log(lib_size+1), bins=100, color='C0', edgecolor='k', alpha=0.8)
axes[0].set_title('Log Library size per cell')
axes[0].set_xlabel('Total counts / cell')
axes[0].set_ylabel('Number of cells')
axes[0].legend()

# Cells per gene histogram
axes[1].hist(cells_per_gene, bins=100, color='C2', edgecolor='k', alpha=0.8)
axes[1].set_title('Number of cells expressing each gene')
axes[1].set_xlabel('Cells with expression > 0')
axes[1].set_ylabel('Number of genes')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# filter cells by log1p(lib_size) > 3 and subset to highly variable genes
cell_mask = np.log(lib_size + 1) > 3
print(f"Keeping {cell_mask.sum()} / {data.n_obs} cells (log1p(lib_size) > 3)")

data = data[cell_mask].copy()

In [ ]:
slk.pp.treat_effect(
    data,
    label_col = 'pert',
    control_label = 'NTC',
    method = "perturbseq",
    inplace=True,
    key='treat_effect'
)

In [ ]:
model = slk.tl.load_results('../../results/raefish.pth')

In [ ]:
slk.tl.eval_single(model, data, obsm_key="z", uns_key="results")

In [ ]:
data.uns['results']

In [ ]:
slk.pl.plot_r2(data)

In [ ]:
slk.pl.plot_corr(data)

### Fig. 2a — SHERLOCK latent UMAP (RAEFISH), coloured by perturbation
*(source cell 21)*

In [ ]:
p_key = slk.configs.get_config('pert_key')
NTC = slk.configs.get_config('ntc_label')

sub = data[data.obs[p_key] != NTC]

sc.pp.neighbors(sub, n_neighbors=15, use_rep='z')
sc.tl.umap(sub)

import random
n_cats = sub.obs[p_key].nunique()
palette = sns.color_palette("hsv", n_colors=n_cats)
random.shuffle(palette)
sc.pl.umap(sub, color='pert', palette=palette, frameon=False)

In [ ]:
zrc = data.uns['results']['z_rho_corr']

print(f"Spearman ρ = {zrc['spearman_r']:.3f}  (p = {zrc['spearman_p']:.1e})")
print(f" Pearson  r = {zrc['pearson_r']:.3f}  (p = {zrc['pearson_p']:.1e})") 

In [ ]:
slk.pl.plot_zcorr(data)

In [ ]:
W = data.uns['results']['W']
plt.hist(W.flatten(), bins=30)
plt.show()

In [ ]:
W_bin = (W > 0.5).astype(float)
sns.heatmap(W, cmap="coolwarm")

In [ ]:

dataset = slk.tl.PerturbMatchingDataset(data)
all_genes = dataset.perturbation_dict.keys()

### Fig. 2b — hierarchical clustering of perturbations (dashed line = cut)
*(source cell 27)*

In [ ]:
groups = slk.tl.cluster_rho(data, t=1.5, show=True)

In [ ]:
#for a non subsetted data, pass in genes corresponding to idx ie dataset.perturbation_dict.keys()
named_groups = slk.tl.group2name(groups, all_genes)
named_groups

### Fig. 2c — SHERLOCK latent UMAP coloured by perturbation group
*(source cell 29)*

In [ ]:
# map perturbation -> group using named_groups (index=gene, column='group')
gene_to_group = named_groups['group'].to_dict()

# assign group to obs based on slk pert key
sub.obs['pert_group'] = sub.obs[p_key].map(gene_to_group)

# mark NTC / unknown perturbations as -1 and ensure integer dtype
sub.obs['pert_group'] = sub.obs['pert_group'].fillna(-1).astype('category')

# quick summary
print(sub.obs['pert_group'].value_counts().sort_index())

sc.pl.umap(sub, color=['pert_group', 'dataset'], title='Perturbation Groups', palette='tab20', size=30)

> **FLAG — needs further investigation.** **Fig. 2a–c can't run as-is**: `results/raefish.pth` (Nov 2025) was saved by an older version of the model code and fails to load (state_dict key mismatch). To regenerate, retrain with cell 14 of `raefish_run.ipynb`. Note that its cell 15 saves over `results/raefish.pth`, which isn't tracked by git, so back the file up first if you want to keep it. Panels from a retrained model won't match the manuscript's exactly.

> **FLAG — needs further investigation.** **Fig. 2d**: GO enrichment of the RAEFISH perturbation groups. No GO-enrichment cell exists in the current notebooks (cell 32 of `raefish_run.ipynb` only prints each group's genes).

> **FLAG — needs further investigation.** **Fig. 2e–h**: EGFR inhibitor sci-Plex screen (Giglio et al.): UMAPs by drug and group, correlation matrix, dendrogram. No notebook for this dataset exists on this branch or any other branch searched.

> **FLAG — needs further investigation.** **Fig. 2i–j**: the manuscript panels have a group colour bar and a green dotted cut line, and Methods describe a metacell CRISPRi dataset (71 guides, latent 16, rank 6, L0 35). The closest current source, `gbm_run.ipynb`, is included below; confirm it is the same model and data. `reproducibility_gbm.ipynb` (cell 34, consensus dendrogram) is the other candidate.

## Source: `notebooks/gbm/gbm_run.ipynb`

Cells copied verbatim (source cell indices): 0, 1, 2, 3, 4, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19. Training (5) and saving (6) are not copied; cell 7 loads `results/gbm_cripsri.pth`.

In [ ]:
# [reproducibility glue] run from the source notebook's directory so its relative paths resolve
os.chdir(os.path.join(REPRO_DIR, '../notebooks/gbm'))
if 'slk' in globals():
    slk.configs.reset_config()

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import sys

import seaborn as sns
import matplotlib.pyplot as plt

import torch

#Make sure to path to wherever sherlock is located
sys.path.append('../../..')
import sherlock as slk

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
slk.configs.reset_config()
slk.configs.show()
print('\n')
slk.configs.set_config('untreated_label', '0.0')
slk.configs.show()

In [ ]:
data = sc.read_h5ad("../../datasets/gbm/filtered_gbm_crispri_crop.h5ad")
data.obs.dose = data.obs.dose.astype(str)
slk.pp.slk_prepare_data(data, pert_key="gene_id", ntc_label="NTC", treatment_key='dose', untreated_label='0.0', inplace=True, force=True)

p_key = slk.configs.get_config('pert_key')
NTC   = slk.configs.get_config('ntc_label')
c_key = slk.configs.get_config('treatment_key')
c_ntc_key = slk.configs.get_config('untreated_label')

all_perts    = data.obs[p_key].unique()
print(f"Cells: {data.n_obs}   Genes: {data.n_vars}")
print(f"Single perts: {len(all_perts)}   NTC cells: {(data.obs[p_key]==NTC).sum()}")

all_conds = data.obs[c_key].unique()
print(f"Conditions: {all_conds}")
print(f"Condition counts:\n{data.obs[c_key].value_counts()}")
print(data.obs[data.obs[c_key] == c_ntc_key].shape)

In [ ]:
slk.pp.treat_effect(
    data,
    label_col = 'pert',
    control_label = 'NTC',
    method = "perturbseq",
    inplace=True,
    key='treat_effect'
)

In [ ]:
model = slk.tl.load_results('../../results/gbm_cripsri.pth')

In [ ]:
slk.tl.eval_single(model, data, obsm_key="z", uns_key="results", device=device)
print(list(data.uns['results'].keys()))

In [ ]:
slk.pl.plot_r2(data)

### Fig. 2i (closest source) — learned perturbation correlation, CRISPRi
*(source cell 10)*

In [ ]:
slk.pl.plot_corr(data)

In [ ]:
p_key = slk.configs.get_config('pert_key')
NTC = slk.configs.get_config('ntc_label')

sub = data[data.obs[p_key] != NTC]

sc.pp.neighbors(sub, n_neighbors=15, use_rep='z')
sc.tl.umap(sub)

import random
n_cats = sub.obs[p_key].nunique()
palette = sns.color_palette("hsv", n_colors=n_cats)
random.shuffle(palette)
sc.pl.umap(sub, color='pert', palette=palette, frameon=False)

In [ ]:
sc.pl.umap(sub, color='treatment', palette='tab20', frameon=False)

In [ ]:
sub.obs['dose_ct'] = sub.obs.treatment.astype(float)
sc.pl.umap(sub, color='dose_ct', frameon=False)

In [ ]:
from scipy.stats import pearsonr, spearmanr

z_corr   = data.uns['results']['z_corr']
rho_corr = data.uns['results']['rho_corr']

pearson  = pearsonr(z_corr.flatten(),  rho_corr.flatten())
spearman = spearmanr(z_corr.flatten(), rho_corr.flatten())
print(f"Spearman ρ = {spearman.statistic:.3f}  (p = {spearman.pvalue:.1e})")
print(f" Pearson  r = {pearson.statistic:.3f}  (p = {pearson.pvalue:.1e})")

In [ ]:
slk.pl.plot_zcorr(data)

In [ ]:
W = data.uns['results']['W']   # (P, latent_dim)
print(f"W shape: {W.shape}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(W.flatten(), bins=40, edgecolor='k', linewidth=0.3)
axes[0].set_title('Gate W distribution')
axes[0].set_xlabel('W value')

W_bin = (W > 0.5).astype(float)
im = axes[1].imshow(W_bin, aspect='auto', cmap='Greys', interpolation='nearest')
axes[1].set_title('Binarised W  (P × d)')
axes[1].set_xlabel('Latent dim')
axes[1].set_ylabel('Perturbation')
plt.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.show()

print(f"Avg active dims per pert: {W_bin.sum(1).mean():.1f} / {W.shape[1]}")

In [ ]:
rho_perts = data.uns['results']['rho_perts']
C = z_corr
print(f"rho_corr: {C.shape}  |  perts: {len(rho_perts)}")

In [ ]:
np.where(rho_perts == 'PDGFRA')

### Fig. 2j (closest source) — hierarchical clustering of CRISPRi perturbations
*(source cell 19)*

In [ ]:
groups       = slk.tl.cluster_rho(data, t=2, rho_corr=C, show=True)
named_groups = slk.tl.group2name(groups, list(rho_perts))
named_groups